# Notebook #1: Data Cleaning & Compilation

## 1.0 Project Dependencies

### 1.1 Pip Installs

In [1]:
# Import pip installs here

### 1.2 Import Libraries

In [2]:
import os
import re
import glob
import pandas as pd

<hr>

## 2.0 Import Editions Datasets

[Link to Microsoft 365 Data Folder](https://americanlibraryassociation.sharepoint.com/sites/UMSIAlternativeSpringBreakProgram/Shared%20Documents/Forms/AllItems.aspx?e=5%3Aeb221561308a4030bf98da546c1fac9b&sharingv2=true&fromShare=true&at=9&CID=a81b58d2%2Dff57%2D4471%2D845b%2Dcf25be35f267&FolderCTID=0x01200076BD73CBF211EE42AA9BAE91D9B6A77D&id=%2Fsites%2FUMSIAlternativeSpringBreakProgram%2FShared%20Documents%2FGeneral%2FUMSI%20Project%201%20%2D%20Editions%20ACRL) 

### 2.1 Editions e-Blasts

In [3]:
editions_eblasts = pd.read_csv("Raw Datasets/Editions eBlasts.csv")

# Set "Name" and "Subject" Columns to be String dtype
editions_eblasts["Name"].astype(str)
editions_eblasts["Subject"].astype(str)


# Set "Sent Column with DateTime
editions_eblasts["Sent"] = pd.to_datetime(editions_eblasts["Sent"])

# Load into Cleaned Datasets Folder
editions_eblasts.to_parquet("Cleaned Datasets/editions_ala_eblasts.parquet", index=False)

### 2.2 Define Function to Load Multi-Sheet CSV Data for Editions SS25 & FW26

In [4]:
def load_google_analytics_csvs(folder: str, season_prefix: str, output_name: str) -> pd.DataFrame:
    
    """
    Reads all CSVs for a given season from a folder.
    - Matches files by season_prefix (e.g. 'Editions catalog SS25')
    - Assigns week=0 for cumulative, week=1,2,3... for weekly files
    - Headers expected on row 10 (index 9)
    - Start/End dates extracted from rows 8 & 9 (index 7 & 8)
    - Saves one combined parquet to /Cleaned Datasets/
    """

    # Find all matching files
    all_files = glob.glob(f"{folder}/{season_prefix}*.csv")

    # Separate cumulative from weekly
    cumulative = [f for f in all_files if "cumulative" in f.lower()]
    weeklies   = sorted([f for f in all_files if re.search(r"week\s*\d+", f, re.IGNORECASE)])

    ordered_files = cumulative + weeklies  # cumulative first → week=0

    dfs = []
    for week_num, filepath in enumerate(ordered_files):
        # Extract dates
        meta = pd.read_csv(filepath, header=None, nrows=10)
        start_raw = str(meta.iloc[7, 0])
        end_raw   = str(meta.iloc[8, 0])
        start_date = pd.to_datetime(re.search(r"\d{8}", start_raw).group(), format="%Y%m%d")
        end_date   = pd.to_datetime(re.search(r"\d{8}", end_raw).group(),   format="%Y%m%d")

        # Load data
        df = pd.read_csv(filepath, header=9)
        df.columns = (
            df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_")
        )

        df["week"]       = week_num
        df["start_date"] = start_date
        df["end_date"]   = end_date

        df = df[df['page_path_and_screen_class'].str.len() < 50]

        print(f"  week={week_num} | '{filepath.split('/')[-1]}' | {start_date.date()} to {end_date.date()} | {len(df)} rows")
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    out_path = f"Cleaned Datasets/{output_name}.parquet"
    combined.to_parquet(out_path, index=False)
    print(f"  Saved → {out_path}\n")
    return combined


# Run for each season
print("Loading SS25...")
ss25 = load_google_analytics_csvs(
    folder="Raw Datasets/Editions SS25",
    season_prefix="Editions catalog SS25",
    output_name="editions_ala_ss25_google"
)

print("Loading FW26...")
fw26 = load_google_analytics_csvs(
    folder="Raw Datasets/Editions FW26",
    season_prefix="Editions catalog FW26",
    output_name="editions_ala_fw26_google"
)

Loading SS25...
  week=0 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(cumulative).csv' | 2025-04-06 to 2025-09-20 | 20 rows
  week=1 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 1).csv' | 2025-04-06 to 2025-04-12 | 17 rows
  week=2 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 10).csv' | 2025-06-08 to 2025-06-14 | 17 rows
  week=3 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 11).csv' | 2025-06-15 to 2025-06-21 | 15 rows
  week=4 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 12).csv' | 2025-06-22 to 2025-06-28 | 17 rows
  week=5 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 13).csv' | 2025-06-29 to 2025-07-05 | 17 rows
  week=6 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 14).csv' | 2025-07-06 to 2025-07-12 | 16 rows
  week=7 | 'Editions catalog SS25_Google Analytics report - Feb 25 to Aug 25(week 15).

### 2\.3 Stats ALA Editions E\-Blasts with Email Body Text

In [5]:
editions_ala_stats = pd.read_csv("Raw Datasets/Stats ALA Editions E-Blasts.csv", encoding='cp1252')

# Load into Cleaned Datasets Folder
editions_ala_stats.to_parquet("Cleaned Datasets/editions_ala_stats_eblasts", index=False)

<hr>

## 3.0 Import ACRL Datasets

### 3.1 ACRL Syllabus Newsletters

In [6]:
acrl_syllabus_new = pd.read_csv("Raw Datasets/Syllabus Newsletters updated 3 4 2026(Syllabus Newsletters).csv")
acrl_syllabus= acrl_syllabus_new.dropna()

# Set dtypes for Non-Float, Non-DateTime Columns
dtypes_mapping = {
    "Name": "str",
    "Folder": "str",
    "# Delivered": "int",
    "Total Opens": "int",
    "Total Clicks": "int"
}

# Set "Sent" Column to DateTime
acrl_syllabus["Sent"] = pd.to_datetime(acrl_syllabus["Sent"])

# Correct Percent Columns' Formatting
percent_cols = ["Open %", "Click %"]
for col in percent_cols:
    acrl_syllabus[col] = (
        acrl_syllabus[col]
        .str.replace("%", "", regex=False)
        .astype(float)
    )

# Load into Cleaned Datasets Folder
acrl_syllabus.to_parquet("Cleaned Datasets/acrl_syllabus.parquet", index=False)
acrl_syllabus

/tmp/ipykernel_1612/1111395860.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acrl_syllabus["Sent"] = pd.to_datetime(acrl_syllabus["Sent"])
/tmp/ipykernel_1612/1111395860.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acrl_syllabus[col] = (
/tmp/ipykernel_1612/1111395860.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-d

,Name,Subject,Sent,Folder,# Delivered,Total Opens,Open %,Total Clicks,Click %
0,Syllabus - February 2026,The Syllabus From ACRL - February 2026,2026-02-18 12:14:00,ACRL,22354.0,5054.0,13.80,696.0,13.80
1,Syllabus - January 2026,The Syllabus From ACRL - January 2026,2026-01-22 11:12:00,ACRL,22456.0,4852.0,21.60,646.0,13.30
2,Syllabus - December 2025,The Syllabus From ACRL - December 2025,2025-12-10 12:13:00,ACRL,22591.0,5108.0,22.60,670.0,13.10
3,Syllabus - November 2025,The Syllabus From ACRL - November 2025,2025-11-20 12:14:00,ACRL,22682.0,5371.0,23.70,624.0,11.62
4,Syllabus - October 2025,The Syllabus From ACRL - October 2025,2025-10-15 12:13:00,ACRL,22775.0,4938.0,21.70,596.0,12.10
5,Syllabus - September 2025,The Syllabus From ACRL - September 2025,2025-09-17 14:38:00,ACRL,22638.0,5196.0,23.00,489.0,9.40
6,Syllabus - August 2025,The Syllabus From ACRL - August 2025,2025-08-20 12:12:00,ACRL,22725.0,8143.0,35.83,535.0,6.60
7,Syllabus - July 2025,The Syllabus From ACRL - July 2025,2025-07-16 12:14:00,ACRL,22793.0,8791.0,38.60,623.0,7.10
8,Syllabus - June 2025,The Syllabus From ACRL - June 2025,2025-06-18 14:40:00,ACRL,22797.0,6302.0,27.60,926.0,7.16
9,Syllabus - May 2025,The Syllabus From ACRL - May 2025,2025-05-21 12:13:00,ACRL,22797.0,6302.0,27.60,700.0,11.11


### 3.2 ACRL Individual Book Promotional Emails


In [7]:
acrl_indiv_emails = pd.read_csv("Raw Datasets/NEW ACRL Indiv Book eBlast Statistics.csv")

# Set "Sent" Column to DateTime
acrl_indiv_emails["Sent"] = pd.to_datetime(acrl_indiv_emails["Sent"])

# Format Percentage Columns Correctly
percent_cols = ["Open %", "Click %", "Unsubs %"]
for col in percent_cols:
    acrl_indiv_emails[col] = (
        acrl_indiv_emails[col]
        .astype(float)
        * 100
    ).round(1)

# Load into Cleaned Datasets Folder
acrl_indiv_emails.to_parquet("Cleaned Datasets/acrl_indiv_emails", index=False)

acrl_indiv_emails

/tmp/ipykernel_1612/461644097.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  acrl_indiv_emails["Sent"] = pd.to_datetime(acrl_indiv_emails["Sent"])


,Name,Subject,Sent,Folder,# Delivered,Delivered %,Total Opens,Open %,Total Clicks,Click %,Unsubs,Unsubs %
0,Library Publishing Book,"New from ACRL - ""Library Publishing: How to La...",2025-10-29 11:14:00,ACRL,18350,99.8,6137,33.4,247,4.0,4,0.0
1,Navigating Disability book,"New from ACRL - ""Navigating Disability in the ...",2025-10-09 11:14:00,ACRL,18408,99.8,6010,32.6,292,4.9,7,0.0
2,MVBD book,"New from ACRL - ""Making Values-Based Decisions...",2025-09-16 11:14:00,ACRL,18211,99.9,5948,32.7,267,4.5,1,0.0
3,Valuing the CCL book,"New from ACRL - ""Valuing the Community College...",2025-07-30 11:14:00,ACRL,18314,99.9,6978,38.1,226,3.2,5,0.0
4,Supporting Neurodiverse book,"New from ACRL - ""Supporting Neurodiverse Stude...",2025-06-24 11:14:00,ACRL,18420,99.9,7672,41.7,354,4.6,10,0.1
5,Open Science Cookbook,"New from ACRL - ""The Open Science Cookbook""",2025-06-11 11:14:00,ACRL,18217,99.9,7833,43.0,686,8.8,0,0.0
6,Legislative Advocacy Book,"New from ACRL - ""Legislative Advocacy and Publ...",2025-05-22 11:14:00,ACRL,18295,99.7,7129,39.0,957,13.4,11,0.1
7,Instructional Design Book,"New from ACRL - ""Instructional Design for Teac...",2025-04-15 11:13:00,ACRL,18142,99.9,5069,27.9,395,7.8,5,0.0
8,Integration Book,"New from ACRL - ""From Interrogation to Integra...",2025-04-09 11:14:00,ACRL,18153,99.9,5276,29.1,239,4.5,4,0.0
9,Student Success Book,"New from ACRL - ""Student Success Librarianship...",2025-03-11 11:14:00,ACRL,17982,99.9,5212,29.0,560,10.7,3,0.0


### 3\.2\.1 Combine Both ACRL Datasets

In [8]:
acrl_all_emails = pd.concat([acrl_indiv_emails, acrl_syllabus], axis=0)
acrl_all_emails = acrl_all_emails.sort_values(by='Sent', ascending=True)

# Load into Cleaned Datasets Folder
acrl_all_emails.to_parquet("Cleaned Datasets/acrl_all_emails.parquet", index=False)

### 3.3 Create ACRL Google Analytics CSV Dataset from PDF

In [9]:
acrl_google_data = {
    "date_start":    ["2025-01-01"],
    "date_end":      ["2026-02-19"],
    "total_views":   [4784],
    "active_users":  [3345],
    "views_per_active_user": [1.43],
    "avg_engagement_time_seconds_per_active_user": [47],
    "event_count":   [13754],
    "key_events":    [0],
    "total_revenue": [0]
}

df_acrl_google = pd.DataFrame(acrl_google_data)
df_acrl_google.to_csv("Raw Datasets/ACRL Google Analytics.csv", index=False)

# Load into Cleaned Datasets Folder
df_acrl_google.to_parquet("Cleaned Datasets/acrl_google", index=False)


### 3.4 Add Highest & Lowest ACRL Email Body Text CSV to `ACRL Indiv Book Promotional Emails.csv`

In [10]:
# ── 1. Body text mapping ───────────────────────────────────────────────────────
body_texts = {
    "CLIPP 48 Book": """ACRL announces the publication of Supporting Diversity through Collection Evaluation, Development, and Weeding: CLIPP #48, compiled and authored by Erika Barber, Julia Bauder, Micki Bichneuk, Chris Jones, Kayla Reed, and Elizabeth Rodrigues. This new book offers information on, and examples for, incorporating diversity concerns and policies into collection management practices.
The College Library Information on Policy and Practice (CLIPP) book series from ACRL provides college and small university libraries analysis and examples of library practices and procedures. Supporting Diversity through Collection Evaluation, Development, and Weeding contains a thorough literature review and bibliography, analysis and discussion of survey results, and sample library collection development policies; diversity, equity, and justice statements; and a harmful content policy.
There are emerging practices for evaluating collection diversity and for diversifying collections via acquisitions, but the question of how to weed in a way that protects and enhances collection diversity has not been thoroughly explored. CLIPP #48 provides a comprehensive survey of how diversity-enhancing collection management practices have filtered into the day-to-day work of average small and medium-sized academic libraries and offers replicable models for library workers.
Supporting Diversity through Collection Evaluation, Development, and Weeding: CLIPP #48 is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the US or (770) 442-8633 for international customers.
Also From ACRL… Creating an Inclusive Library: Approaches for Increasing Engagement and Use with Students of Color Rethinking Institutional Repositories: Innovations in Management, Collections, and Inclusion""",

    "Legislative Advocacy Book": """ACRL announces the publication of Legislative Advocacy and Public Policy Work for Academic and Research Library Workers, edited by Raymond Pun, Sonya M. Durney, and Tarida Anantachai, addressing academic library workers' roles in influencing legislative and public policies and how to make meaningful impacts.
Library advocacy is a deliberate, planned, and sustained effort to develop understanding and support for libraries and the communities they serve. Advocacy is not optional—it is a vital responsibility to protect the integrity and impact of libraries in society.
Legislative Advocacy and Public Policy Work for Academic and Research Library Workers is a comprehensive resource addressing our specific roles in influencing the various legislative and public policies that directly impact academic libraries, our academic communities, and the library community at large. It explores the ways academic and research library workers are engaging in advocacy, from testifying at governmental levels to collaborating with diverse constituents and legislators to bringing our academic advocacy efforts to discussions within state and regional library associations.
As educators and researchers, library workers bring invaluable expertise to the legislative table. We can help ensure that decisions affecting academic and research libraries and their communities are informed, equitable, and aligned with the values of intellectual freedom and lifelong learning. Legislative Advocacy and Public Policy Work for Academic and Research Library Workers offers ways to make meaningful legislative and policy impacts.
Legislative Advocacy and Public Policy Work for Academic and Research Library Workers: Perspectives and Strategies is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the US or (770) 442-8633 for international customers.
Also From ACRL… Innovative Library Workplaces: Transformative Human Resource Strategies Managing Crises in the Academic Library: Past, Present, and Future""",

    "Open Science Cookbook": """ACRL announces the publication of The Open Science Cookbook, edited by Emily Bongiovanni, Melanie Gainey, Chaz Griego, and Lencia McKee, a collection of lesson plans and activities for supporting openly accessible, reproducible research.
Open science promotes more transparent, accessible, and reproducible research and extends beyond the sciences, fostering this inclusivity across all disciplines. There are many benefits to practicing open science, including opportunities for interdisciplinary collaboration, increased visibility and impact, and enhanced reusability of research.
The Open Science Cookbook provides a wide variety of lesson plans and learning activities for supporting collaborative, transparent, openly accessible, and reproducible research. In five sections, it has something for beginners to more advanced practitioners and for different audience sizes.
Program Development Instruction Outreach Events Collaborations and Partnerships
Just as freely sharing data and workflows enables key breakthroughs in major fields, sharing open science practices and resources creates an even stronger foundation for this necessary growth at institutions around the world. The Open Science Cookbook offers innovative ways for academic libraries to promote open science through advocacy and education.
The Open Science Cookbook is available for purchase in print through the ALA Online Store and Amazon.com; by telephone order at (800) 621-2736 or (773) 702-7010; and as an open access edition.
Also From ACRL… Rethinking Institutional Repositories: Innovations in Management, Collections, and Inclusion Scholarly Communication Librarianship and Open Knowledge""",

    "Small to Mid Libraries Book": """ACRL announces the publication of The Small to Mid-Size Academic Library: Collaborations and Outreach, edited by Camille Abdeljawad and Danielle Theiss. This book—the first in a new series from ACRL—captures how academic library workers at these institutions are providing engagement and outreach opportunities for students by partnering with other entities across their campuses.
Small to mid-size academic libraries often operate with reduced staffing, smaller budgets, and competing priorities. These limitations reduce the resources that can be allocated to programming and outreach efforts.
In three parts, The Small to Mid-Size Academic Library: Collaborations and Outreach provides easy-to-implement ideas and strategies for course, campus, and community outreach.
Part 1: Collaborations Cross-Campus Initiatives External Collaborations Part 2: Academic Success Initiatives Writing and Composition Orientation and Programming Part 3: Evolving Roles of Libraries in Student Success Student Wellness Emerging Roles for Librarians
Increased student engagement with cocurricular library and cross-departmental activities can lead to higher student retention and persistence rates. Academic libraries have an important role to play in providing these opportunities, and the volume provides effective practices for supporting student success.
The Small to Mid-Size Academic Library: Collaborations and Outreach is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the US or (770) 442-8633 for international customers.
Also From ACRL… Creating an Inclusive Library: Approaches for Increasing Engagement and Use with Students of Color Empathy by Design: Empathy-Driven Marketing for Libraries""",

    "Student Success Book": """ACRL announces the publication of Student Success Librarianship: Critical Perspectives on an Evolving Profession, edited by Melody Lee Rood and Olivia Patterson. The book examines this emerging position through a critical lens and provides insight and advice to help the profession work toward a positive evolution of this important role.
Student success librarian positions, while increasingly common in academic libraries, are still so new to the field that many librarians with this title are the first person in the position and their roles are ambiguous and difficult to define. They often include quantifiable metrics such as retention rates, academic persistence, and graduation rates, as well as elements like student well-being, belonging, and a sense of purpose. This broad scope has intensified already challenging and emotionally taxing work.
In three sections — Theory, Praxis, and Research — Student Success Librarianship explores how to articulate, set boundaries for, and bring our humanity to the role; address student mental health and provide multilingual resources and support; and map both the current state of student success and a vision for its future.
Student success librarians can feel alone in both their love for and critiques of their profession and their roles within it. Student Success Librarianship can help you celebrate your work, find areas of improvement, and offers validation and inspiration.
Student Success Librarianship: Critical Perspectives on an Evolving Profession is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the US or (770) 442-8633 for international customers.
Also From ACRL… Creating an Inclusive Library: Approaches for Increasing Engagement and Use with Students of Color Supporting Student Parents in the Academic Library: Designing Spaces, Policies, and Services""",

    "Empathy Book": """ACRL announces the publication of Empathy by Design: Empathy-Driven Marketing for Libraries by Sabine Jean Dantus. The book offers step-by-step strategies for understanding why people visit the library and tailoring your marketing with personalization that resonates with users on a deeper level.
The library is a universal resource where knowledge and information meet. To advertise this resource and advance equal access in positive ways, libraries must develop strategies, campaigns, and messages that show they care about the lives of their diverse communities.
Empathy by Design: Empathy-Driven Marketing for Libraries provides real-world solutions for understanding your target audience through empathy and demonstrates how to gather and use data to develop messages and programming that fosters meaningful connections and engagement. You'll find ideas for understanding the customer journey, creating an empathic library brand, and creating empathy-driven marketing strategies, campaigns, content, and tactics.
Today's library marketers should both understand the effectiveness of using empathy in marketing and use it as a radical tool for advancing our profession's values of diversity, equity, inclusion, and accessibility. The strategies outlined in Empathy by Design can give you the tools you need to make your marketing—and your library—more targeted and empathic.
Empathy by Design: Empathy-Driven Marketing for Libraries is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the U.S. or (770) 442-8633 for international customers.
Also From ACRL… Designing Libraries for the 21st Century Integrated Library Planning: A New Model for Strategic and Dynamic Planning, Management, and Assessment""",

    "Pred Pub Book": """ACRL announces the publication of Predatory Publishing and Global Scholarly Communications by Monica Berger, book number 81 in the Publications in Librarianship series. This thorough book provides tools for understanding and teaching the impact of predatory publishing and contributing to its mitigation in the context of advocating for a more balanced, sustainable, and humane scholarly communications ecosystem.
Predatory publishing is a complex problem that harms a broad array of stakeholders and concerns across the scholarly communications system. It shines a light on the inadequacies of scholarly assessment and related rewards systems, contributes to the marginalization of scholarship from less developed countries, and negatively impacts the acceptance of open access.
To fix what is broken in scholarly communications, academic librarians must act as both teachers and advocates and partner with other stakeholders who have the agency to change how scholarship is produced, assessed, and rewarded. Predatory Publishing and Global Scholarly Communications is a unique and comprehensive exploration of predatory publishing in four parts:
Background Characteristics and Research The Geopolitics of Scholarly Publishing Responses and Solutions
It examines the history of predatory publishing and basics of scholarly assessment; identifies types of research misconduct and unethical scholarly behaviors; provides critical context to predatory publishing and scholarly communications beyond the Global North; and offers structural and pedagogical solutions and teaching materials for librarians to use in their work with authors, students, faculty, and other stakeholders.
Predatory Publishing and Global Scholarly Communications gives powerful insight into predatory publishing worldwide, inside and outside of the library community, and provides methods for understanding and teaching its impact and contributing to its improvement.
Predatory Publishing and Global Scholarly Communications is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the U.S. or (770) 442-8633 for international customers.
Also From ACRL… Digital Humanities in the Library, Second Edition Rethinking Institutional Repositories: Innovations in Management, Collections, and Inclusion""",

    "Spatial Lit Book": """ACRL announces the publication of Spatial Literacy in Public Health: Faculty-Librarian Teaching Collaborations, edited by Laureen P. Cantwell-Jurkovic and Tammy P. Parece, a collection of ideas and plans for collaborative spatial literacy teaching and learning initiatives focusing on geographic information systems (GIS)-based and GIS-related instruction through the lens of public health topics.
Spatial literacy — the ability to visualize, understand, and use the properties of space to communicate, reason, and problem-solve — is relevant across a wealth of disciplines. Spatial Literacy in Public Health offers step-by-step learning activities, teaching tips, recommended readings, and four-color maps as well as other useful illustrations. These not only support student learning, but also professional development for librarians interested in spatial literacy instruction and in pitching such instruction to potential faculty collaborators. Interdisciplinary topics include supply chain management, social media campaigns, data visualization, racial disparities, and other demographic themes related to immunization patterns, epidemiology, recreation access, and community health and environmental science.
Spatial Literacy in Public Health offers specific plans for collaborative, interdisciplinary spatial literacy instruction and activities. Chapters also connect with supplementary content in the ACRL Framework for Information Literacy, including activity-focused slide decks and more, all findable with the tag #SpatialLit and freely available. Learning activities support open-source tools, open educational resources (OER), and other institutional cost-saving initiatives, and the ability to adapt from in-person to virtual instruction settings.
Spatial Literacy in Public Health: Faculty-Librarian Teaching Collaborations is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the U.S. or (770) 442-8633 for international customers.
Also From ACRL… Practicing Privacy Literacy in Academic Libraries Unframing the Visual: Visual Literacy Pedagogy in Academic Libraries and Information Spaces""",

    "Unframing Visual Book": """ACRL announces the publication of Unframing the Visual: Visual Literacy Pedagogy in Academic Libraries and Information Spaces, edited by Maggie Murphy, Stephanie Beene, Katie Greer, Sara Schumacher, and Dana Statton Thompson. This collection provides new ideas and inspiration for how to unframe, adapt, and apply visual literacy pedagogy and praxis in your work.
Visual literacy is an interconnected set of practices, habits, and values for participating in visual culture that can be developed through critical, ethical, reflective, and creative engagement with visual media. Approaches to teaching visual literacy in higher education must include a focus on context and not just content, process and not just product, impact and not just intent. Unframing is an approach to visual literacy pedagogy that acknowledges that visuals are a pervasive part of everyday life, as well as embedded into every scholarly discipline.
In four parts, Unframing the Visual: Visual Literacy Pedagogy in Academic Libraries and Information Spaces explores:
Participating in a Changing Visual Information Landscape Perceiving Visuals as Communicating Information Practicing Visual Discernment and Criticality Pursuing Social Justice through Visual Practice
Twenty-four full color chapters present a range of theoretical and practical approaches to visual literacy pedagogy that illustrate, connect with, extend, and critique concepts from the Framework for Visual Literacy in Higher Education: Companion Document to the Framework for Information Literacy for Higher Education. Topics include using TikTok to begin a conversation on academic honesty and marginalization; supporting disciplines to move to multimodal public communication assignments; critical data visualization; and exclusionary practices in visual media.
Unframing the Visual: Visual Literacy Pedagogy in Academic Libraries and Information Spaces is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; and by telephone order at (866) 746-7252 in the US or (770) 442-8633 for international customers.
Also From ACRL… Comic Books, Special Collections, and the Academic Library Embracing Change: Alternatives to Traditional Research Writing Assignments""",

    "Valuing the CCL book": """ACRL announces the publication of Valuing the Community College Library: Impactful Practices for Institutional Success, edited by Terra Jacobson and Spencer Brayton, a collection of activities and ideas that showcase the unique value that community college libraries bring to their institutions and how to capture and communicate that value.
Sharing the work of our community college libraries—our efforts, stories, and how we're advancing the institution—and advocating to our peers and administration can boost the libraries' role on our singular campuses.
Valuing the Community College Library provides a holistic approach to exhibiting community college library value through historical context, practical applications, and future thinking. Through case studies, editorials from administrators, and practical approaches, it addresses why community college libraries exist and should exist, and the nuanced approaches to how library workers situate themselves at their institutions.
Community college libraries need to provide access to content, people, space, and technology and offer instruction, but can also serve as an outreach arm in advancing the mission of open enrollment and affordable access to higher education. Valuing the Community College Library can help you be an advocate for your library on campus and in your community.
Valuing the Community College Library: Impactful Practices for Institutional Success is available for purchase in print and as an ebook through the ALA Online Store; in print through Amazon.com; via EBSCO, ProQuest, and other ebook vendors; and by telephone order at (866) 746-7252 in the U.S. or (770) 442-8633 for international customers.
Also From ACRL… The Community College Library: Collections and Technical Services Supporting Neurodiverse Students in Academic Libraries"""
}

# ── 2. Map body texts and label high/low ──────────────────────────────────────
# Tell me which 5 are high and which 5 are low and I'll fill these in
high_performers = [
    "CLIPP 48 Book",
    "Legislative Advocacy Book",
    "Open Science Cookbook",
    "Small to Mid Libraries Book",
    "Student Success Book"
]

low_performers = [
    "Empathy Book",
    "Pred Pub Book",
    "Spatial Lit Book",
    "Unframing Visual Book",
    "Valuing the CCL book"
]

acrl_indiv_emails["body_text"] = acrl_indiv_emails["Name"].map(body_texts)
acrl_indiv_emails["group"] = acrl_indiv_emails["Name"].apply(
    lambda x: "high" if x in high_performers else ("low" if x in low_performers else None)
)

# 3. Clean Subject Lines
def clean_subject(text):
    # Remove replacement character sequences like â€œ, â€, â€™, etc.
    text = re.sub(r'[â€œâ€\x80-\x9f\ufffd]+', '"', text)
    # Clean up any doubled/trailing quotes and whitespace
    text = re.sub(r'"+', '"', text)
    text = re.sub(r'"\s*$', '"', text)
    text = text.replace('â€™', "'")
    return text.strip()

acrl_indiv_emails["Subject_clean"] = acrl_indiv_emails["Subject"].apply(clean_subject)

# 4. Quality Check
acrl_indiv_emails

acrl_indiv_emails.to_parquet("Cleaned Datasets/acrl_indiv_emails", index=False)
print("Saved → Cleaned Datasets/acrl_indiv_emails.parquet")

Saved → Cleaned Datasets/acrl_indiv_emails.parquet
